# Modelling
Initial baseline, Random Forest classifier to predict obesity.

In [1]:
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler


df = pd.read_parquet('brfss_2024_qusipe_clean.parquet')


health_patterns = ['BMI', 'ASTH', 'MICHD', 'RFHLTH', 'PHYS14D', 'MENT14D', 'HLTHPL2', 'DENVST3', 'DIABETES', 'HYPERTENSION', 'CHOLESTEROL']
health_vars = [c for c in df.columns if any(p.lower() in c.lower() for p in health_patterns)]
print("Health variables for EDA:", health_vars)
# Load PCA object
pca = joblib.load('pca_health_vars.joblib')
print("PCA object loaded.")

# Transform new data using the saved PCA
df_pca = pca.transform(df[health_vars])
print("PCA transformed shape:", df_pca.shape)

Health variables for EDA: ['ASTHMA3', 'ASTHNOW', 'BMI', 'BMI_raw', 'CASTHDX2', 'CASTHNO2', '_ASTHMS1', '_BMI5CAT', '_CASTHM1', '_DENVST3', '_HLTHPL2', '_LTASTH1', '_MENT14D', '_MICHD', '_PHYS14D', '_RFBMI5', '_RFHLTH']
PCA object loaded.
PCA transformed shape: (409415, 5)


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


X = df_pca
y = df['obesity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

# Evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_proba)
print("\nROC-AUC Score:", roc_auc)

Confusion Matrix:
[[73721     0]
 [ 8162     0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     73721
           1       0.00      0.00      0.00      8162

    accuracy                           0.90     81883
   macro avg       0.45      0.50      0.47     81883
weighted avg       0.81      0.90      0.85     81883


ROC-AUC Score: 0.5009415104367696


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Random Forest (Baseline Model)
- Captures nonlinear relationships
- Provides feature importance for interpretability with PCA

- Model Evaluation Metrics

  - **Accuracy:** 0.90
  - **Precision (Class 0):** 0.90
  - **Precision (Class 1):** 0.00
  - *It shows class imbalance (currently implementing the SMOTE method)*
